## Phase 2.5 – Manual Cluster Verification and Sensor Selection

### Objective:
The goal of Phase 2.5 is to refine and finalize the list of sensors for each FD00x dataset that will be used in the upcoming classification phase (Phase 3). This step ensures that the selected sensors have clear and distinct patterns across degradation stages and are not redundant or flat. A well-curated sensor set is critical for building a robust and interpretable classifier that can reliably predict degradation stages in real-world engine systems.

### Background:
While Phase 1 applied clustering (KMeans) to assign degradation stage labels and Phase 2 trained preliminary classifiers, this intermediate Phase 2.5 performs a **manual audit** of sensor behavior. Not all sensors carry useful information about engine degradation, and some may show overlapping or non-distinct values across stages. Using such sensors can lead to model confusion, overfitting, or reduced interpretability.

Hence, in Phase 2.5, we perform:

- Sensor variance analysis
- Visualization of sensor behavior across clustered stages
- Δmean (mean difference) analysis between stages for each sensor
- Sensor pruning based on overlapping behavior

This phase ensures scientific rigor and prepares the dataset for high-performance, high-trust modeling.

---

### Step-by-Step Procedure

#### Step 1: Sensor Variance Ranking
- All sensors were first ranked based on their **variance across the entire dataset**.
- Sensors with very low standard deviation (less than 0.001) were dropped immediately as they remain nearly constant over time and contribute no useful signal.
- From the remaining sensors, the **top 5 high-variance sensors** were selected for deeper analysis.

#### Step 2: Sensor Trend Visualization
- For each of the top 5 sensors per FD00x dataset, line plots were generated to visualize the **sensor values over time**, color-coded by the `kmeans_stage` label.
- These plots helped manually inspect whether the sensor measurements increase or decrease meaningfully across the 5 degradation stages.
- If stage transitions were visibly ambiguous or overlapping, the sensor was marked for further inspection.

#### Step 3: Coinciding Stage Detection (Δmean Analysis)
- To formally identify overlaps, the **mean value of each sensor was computed per stage**.
- Pairwise differences (Δmean) between stage combinations were calculated.
- Any sensor with multiple stage pairs showing Δmean less than a threshold (0.01) was considered poorly discriminative and **dropped** from further use.
- This step ensured only those sensors which show clear, distinct values between stages were retained.

#### Step 4: Final Sensor Selection
- After eliminating coinciding sensors, the **final list of sensors** was finalized for each dataset.
- These sensors form the input feature set for the Phase 3 classification models.

---

### Dataset-Wise Summary

#### FD001
- **Initial Top Sensors (Variance):** `['sensor_11', 'sensor_12', 'sensor_4', 'sensor_2', 'sensor_21']`
- **Dropped Due to Overlaps:** `'sensor_21'` (Stage 2 vs 3 had poor separation)
- **Final Sensors for Phase 3:** `['sensor_11', 'sensor_12', 'sensor_4', 'sensor_2']`

#### FD002
- **Initial Top Sensors (Variance):** `['sensor_16', 'sensor_1', 'sensor_19', 'sensor_13', 'sensor_12']`
- **Dropped Due to Overlaps:** `'sensor_16', 'sensor_19', 'sensor_13'`
- **Final Sensors for Phase 3:** `['sensor_1', 'sensor_12']`

#### FD003
- **Initial Top Sensors (Variance):** `['sensor_11', 'sensor_12', 'sensor_7', 'sensor_17', 'sensor_4']`
- **Dropped Due to Overlaps:** None
- **Final Sensors for Phase 3:** `['sensor_11', 'sensor_12', 'sensor_7', 'sensor_17', 'sensor_4']`

#### FD004
- **Initial Top Sensors (Variance):** `['sensor_16', 'sensor_1', 'sensor_19', 'sensor_13', 'sensor_2']`
- **Dropped Due to Overlaps:** `'sensor_16', 'sensor_19', 'sensor_13'`
- **Final Sensors for Phase 3:** `['sensor_1', 'sensor_2']`

---

### Output Artifacts

- Sensor behavior plots saved to:
  - `../figures/manual_cluster_verification/FD001/`
  - `../figures/manual_cluster_verification/FD002/`
  - `../figures/manual_cluster_verification/FD003/`
  - `../figures/manual_cluster_verification/FD004/`

- Final datasets with `final_stage` labels saved as:
  - `corrected_clustered_train_FD001.csv`
  - `corrected_clustered_train_FD002.csv`
  - `corrected_clustered_train_FD003.csv`
  - `corrected_clustered_train_FD004.csv`

---

### Why This Step is Crucial for NASA-Grade Predictive Maintenance

- Avoids inclusion of misleading sensors that can bias classification outcomes.
- Enhances explainability and interpretability of the classification and regression models.
- Strengthens the pipeline by filtering out noise and ensuring signal clarity.
- Aligns with real-world industrial expectations where sensor quality varies and only meaningful signals should inform maintenance decisions.

This completes Phase 2.5 and lays the foundation for confident and high-performance modeling in Phase 3 (Classification of Health Stage).


---
# Step : Load Clustered FD001 Data + Top 5 Sensor Check

In [36]:
import pandas as pd

# Load clustered FD001 dataset
df_fd001 = pd.read_csv("../data/clustered_train_FD001.csv")

# Step 1: Identify all sensor columns
sensor_columns_fd001 = [col for col in df_fd001.columns if 'sensor_' in col]

# Step 2: Drop sensors with extremely low standard deviation
flat_sensors_fd001 = [s for s in sensor_columns_fd001 if df_fd001[s].std() < 0.001]
sensor_columns_fd001 = [s for s in sensor_columns_fd001 if s not in flat_sensors_fd001]

# Step 3: Compute variances
variances_fd001 = df_fd001[sensor_columns_fd001].var().sort_values(ascending=False)

# Step 4: Select top 5 high-variance sensors (for now, include sensor_21)
top_sensors_fd001 = variances_fd001.head(5).index.tolist()

# Final Output
print("Flat / Constant Sensors to drop:", flat_sensors_fd001)
print("Refined Top 5 Variance Sensors in FD001 (initial selection):", top_sensors_fd001)


Flat / Constant Sensors to drop: []
Refined Top 5 Variance Sensors in FD001 (initial selection): ['sensor_11', 'sensor_12', 'sensor_4', 'sensor_2', 'sensor_21']


# Step: Plot Sensor Behavior Across Cluster Stages (FD001)

In [37]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Load clustered FD001 dataset
df_fd001 = pd.read_csv("../data/clustered_train_FD001.csv")

# Top 5 sensors selected in Step 1
top_sensors_fd001 = ['sensor_11', 'sensor_12', 'sensor_4', 'sensor_2', 'sensor_21']

# Create output directory if not exists
output_dir = "../figures/manual_cluster_verification/FD001/"
os.makedirs(output_dir, exist_ok=True)

# Seaborn plot settings
sns.set(style="whitegrid")

# Plot each sensor's trend across kmeans stages
for sensor in top_sensors_fd001:
    plt.figure(figsize=(10, 6))
    sns.lineplot(
        x='time',
        y=sensor,
        hue='kmeans_stage',
        data=df_fd001,
        palette='tab10'
    )
    plt.title(f"{sensor} Behavior Across KMeans Cluster Stages - FD001")
    plt.xlabel("Cycle Number (Time)")
    plt.ylabel(f"{sensor} Reading (Normalized)")
    plt.legend(title="KMeans Stage", loc='best')
    plt.grid(True)
    plt.tight_layout()

    # Save the plot
    filename = f"{sensor}_kmeans_stage_FD001.png"
    filepath = os.path.join(output_dir, filename)
    plt.savefig(filepath)
    plt.close()

print(" Sensor behavior plots saved for FD001.")


 Sensor behavior plots saved for FD001.


# Step: FD001 Coinciding Stage Detection Script

In [ ]:
import pandas as pd
import numpy as np
import itertools

# Load clustered FD001 dataset
df_fd001 = pd.read_csv("../data/clustered_train_FD001.csv")

# Refined top 5 sensors from Step 1
top_sensors_fd001 = ['sensor_11', 'sensor_12', 'sensor_4', 'sensor_2', 'sensor_21']

# Define the delta threshold for considering stage pairs as coinciding
threshold = 0.01
coinciding_pairs = []

print("===== FD001 Coinciding Sensors and Stage Pairs =====")
for sensor in top_sensors_fd001:
    # Compute mean for each stage   
    stage_means = df_fd001.groupby('kmeans_stage')[sensor].mean().sort_index()

    # Compare all stage combinations
    for stage_a, stage_b in itertools.combinations(stage_means.index, 2):
        delta = abs(stage_means[stage_a] - stage_means[stage_b])
        if delta < threshold:
            coinciding_pairs.append((sensor, stage_a, stage_b, delta))
            print(f"⚠️ {sensor}: Stage {stage_a} & Stage {stage_b} → Δmean = {delta:.5f}")

if not coinciding_pairs:
    print(" No coinciding stage pairs found below the threshold.")


===== FD001 Coinciding Sensors and Stage Pairs =====
⚠️ sensor_21: Stage Stage 2 & Stage Stage 3 → Δmean = 0.00953



 # Step : Finalize Top Sensors for Classification (FD001)

In [39]:
import pandas as pd

# Load previously clustered FD001 data
df_fd001 = pd.read_csv("../data/clustered_train_FD001.csv")

# From earlier steps
initial_top_sensors = ['sensor_11', 'sensor_12', 'sensor_4', 'sensor_2', 'sensor_21']
problematic_sensors = ['sensor_21']  # Coinciding stages detected for this sensor

# Final selection — drop problematic ones
final_top_sensors_fd001 = [s for s in initial_top_sensors if s not in problematic_sensors]

print("Initial Top Sensors (based on variance):", initial_top_sensors)
print("Removed due to poor stage separation:", problematic_sensors)
print(" Final Top Sensors for Phase 3 Classification (FD001):", final_top_sensors_fd001)

# Save final sensor list for reuse
pd.Series(final_top_sensors_fd001).to_csv("../data/final_top_sensors_fd001.csv", index=False, header=["sensor"])


Initial Top Sensors (based on variance): ['sensor_11', 'sensor_12', 'sensor_4', 'sensor_2', 'sensor_21']
Removed due to poor stage separation: ['sensor_21']
 Final Top Sensors for Phase 3 Classification (FD001): ['sensor_11', 'sensor_12', 'sensor_4', 'sensor_2']


# Step: Add final_stage and Save

In [45]:
import pandas as pd

# Load the clustered FD001 dataset
df_fd001 = pd.read_csv("../data/clustered_train_FD001.csv")

# Drop sensor_21 due to poor stage separation (as decided in Phase 2.5)
dropped_sensors_fd001 = ['sensor_21']

# Add final_stage column (validated to be same as kmeans_stage after inspection)
df_fd001['final_stage'] = df_fd001['kmeans_stage']

# Save the corrected FD001 dataset for Phase 3
df_fd001.to_csv("../data/corrected_clustered_train_FD001.csv", index=False)

print(" Final version saved as corrected_clustered_train_FD001.csv with 'final_stage' column added.")


 Final version saved as corrected_clustered_train_FD001.csv with 'final_stage' column added.


 ---
 # Step : Load FD002 Clustered Data + Get Top 5 Sensors by Variance

In [40]:
import pandas as pd

# Load clustered FD002 dataset
df_fd002 = pd.read_csv("../data/clustered_train_FD002.csv")

# Identify all sensor columns
sensor_columns_fd002 = [col for col in df_fd002.columns if 'sensor_' in col]

# Drop sensors with near-zero standard deviation (flat sensors)
flat_sensors_fd002 = [s for s in sensor_columns_fd002 if df_fd002[s].std() < 0.001]
sensor_columns_fd002 = [s for s in sensor_columns_fd002 if s not in flat_sensors_fd002]

# Compute variances and get top 5 high-variance sensors
variances_fd002 = df_fd002[sensor_columns_fd002].var().sort_values(ascending=False)
top_sensors_fd002 = variances_fd002.head(5).index.tolist()

# Display summary
print("Flat / Constant Sensors to drop:", flat_sensors_fd002)
print("Refined Top 5 Variance Sensors in FD002 (initial selection):", top_sensors_fd002)


Flat / Constant Sensors to drop: []
Refined Top 5 Variance Sensors in FD002 (initial selection): ['sensor_16', 'sensor_1', 'sensor_19', 'sensor_13', 'sensor_12']


# Step – Plot Behavior of Top Sensors by Cluster Stage (FD002)

In [41]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Load clustered FD002 dataset
df_fd002 = pd.read_csv("../data/clustered_train_FD002.csv")

# Refined top 5 sensors from Step 1
top_sensors_fd002 = ['sensor_16', 'sensor_1', 'sensor_19', 'sensor_13', 'sensor_12']

# Ensure output directory exists
output_dir = "../figures/manual_cluster_verification/FD002/"
os.makedirs(output_dir, exist_ok=True)

# Plot styling
sns.set(style="whitegrid")

# Plot sensor behavior across kmeans stages
for sensor in top_sensors_fd002:
    plt.figure(figsize=(10, 6))
    sns.lineplot(
        x='time',
        y=sensor,
        hue='kmeans_stage',
        data=df_fd002,
        palette='tab10',
        legend='full'
    )
    plt.title(f"{sensor} behavior across KMeans Cluster Stages - FD002")
    plt.xlabel("Cycle Number (Time)")
    plt.ylabel(f"{sensor} Reading (Normalized)")
    plt.legend(title='KMeans Stage')
    plt.tight_layout()
    plt.savefig(f"{output_dir}{sensor}_kmeans_stage_FD002.png")
    plt.close()

print(" Sensor behavior plots saved for FD002.")


 Sensor behavior plots saved for FD002.


# Step: FD002 Coinciding Stage Detection Script

In [42]:
import itertools

# Assume df_fd002 is already loaded and has 'kmeans_stage' column
top_sensors_fd002 = ['sensor_16', 'sensor_1', 'sensor_19', 'sensor_13', 'sensor_12']

coinciding_pairs = []

print("===== FD002 Coinciding Sensors and Stage Pairs =====")
for sensor in top_sensors_fd002:
    stage_means = df_fd002.groupby('kmeans_stage')[sensor].mean().sort_index()
    for stage_a, stage_b in itertools.combinations(stage_means.index, 2):
        delta = abs(stage_means[stage_a] - stage_means[stage_b])
        if delta < 0.01:
            coinciding_pairs.append((sensor, stage_a, stage_b, delta))
            print(f"⚠️ {sensor}: Stage {stage_a} & Stage {stage_b} → Δmean = {delta:.5f}")


===== FD002 Coinciding Sensors and Stage Pairs =====
⚠️ sensor_16: Stage Stage 0 & Stage Stage 2 → Δmean = 0.00000
⚠️ sensor_16: Stage Stage 0 & Stage Stage 4 → Δmean = 0.00000
⚠️ sensor_16: Stage Stage 1 & Stage Stage 3 → Δmean = 0.00000
⚠️ sensor_16: Stage Stage 2 & Stage Stage 4 → Δmean = 0.00000
⚠️ sensor_19: Stage Stage 0 & Stage Stage 1 → Δmean = 0.00000
⚠️ sensor_19: Stage Stage 0 & Stage Stage 3 → Δmean = 0.00000
⚠️ sensor_19: Stage Stage 0 & Stage Stage 4 → Δmean = 0.00000
⚠️ sensor_19: Stage Stage 1 & Stage Stage 3 → Δmean = 0.00000
⚠️ sensor_19: Stage Stage 1 & Stage Stage 4 → Δmean = 0.00000
⚠️ sensor_19: Stage Stage 3 & Stage Stage 4 → Δmean = 0.00000
⚠️ sensor_13: Stage Stage 0 & Stage Stage 1 → Δmean = 0.00004
⚠️ sensor_13: Stage Stage 0 & Stage Stage 3 → Δmean = 0.00027
⚠️ sensor_13: Stage Stage 0 & Stage Stage 4 → Δmean = 0.00010
⚠️ sensor_13: Stage Stage 1 & Stage Stage 3 → Δmean = 0.00023
⚠️ sensor_13: Stage Stage 1 & Stage Stage 4 → Δmean = 0.00006
⚠️ sensor_13: Sta

# Step: Finalize Top Sensors for Phase 3 — Drop Misleading Ones

In [43]:
# Initial top sensors from variance ranking
initial_top_sensors_fd002 = ['sensor_16', 'sensor_1', 'sensor_19', 'sensor_13', 'sensor_12']

# Sensors to drop (based on coinciding stage means)
drop_sensors_fd002 = ['sensor_16', 'sensor_19', 'sensor_13']

# Final sensors to retain for Phase 3 Classification
final_top_sensors_fd002 = [s for s in initial_top_sensors_fd002 if s not in drop_sensors_fd002]

print("Initial Top Sensors (based on variance):", initial_top_sensors_fd002)
print("Removed due to poor stage separation:", drop_sensors_fd002)
print(" Final Top Sensors for Phase 3 Classification (FD002):", final_top_sensors_fd002)


Initial Top Sensors (based on variance): ['sensor_16', 'sensor_1', 'sensor_19', 'sensor_13', 'sensor_12']
Removed due to poor stage separation: ['sensor_16', 'sensor_19', 'sensor_13']
 Final Top Sensors for Phase 3 Classification (FD002): ['sensor_1', 'sensor_12']


# Step: Add final_stage and Save

In [44]:
import pandas as pd

# Load the clustered dataset
df_fd002 = pd.read_csv("../data/clustered_train_FD002.csv")

# Define sensors to drop due to poor separation in Phase 2.5
dropped_sensors = ['sensor_16', 'sensor_19', 'sensor_13']

# Add final_stage column (same as kmeans_stage for now — validated by you)
df_fd002['final_stage'] = df_fd002['kmeans_stage']

# Save corrected and cleaned version for Phase 3 onwards
df_fd002.to_csv("../data/corrected_clustered_train_FD002.csv", index=False)

print(" Final version saved as corrected_clustered_train_FD002.csv with 'final_stage' column added.")


 Final version saved as corrected_clustered_train_FD002.csv with 'final_stage' column added.


---
# Step : Load FD003 Clustered Data + Identify Top 5 Sensors by Variance

In [46]:
import pandas as pd

# Load clustered FD003 dataset
df_fd003 = pd.read_csv("../data/clustered_train_FD003.csv")

# Identify all sensor columns
sensor_columns_fd003 = [col for col in df_fd003.columns if 'sensor_' in col]

# Drop sensors with very low variance
flat_sensors_fd003 = [s for s in sensor_columns_fd003 if df_fd003[s].std() < 0.001]
sensor_columns_fd003 = [s for s in sensor_columns_fd003 if s not in flat_sensors_fd003]

# Compute variance and select top 5
variances_fd003 = df_fd003[sensor_columns_fd003].var().sort_values(ascending=False)
top_sensors_fd003 = variances_fd003.head(5).index.tolist()

print("Flat / Constant Sensors to drop:", flat_sensors_fd003)
print("Refined Top 5 Variance Sensors in FD003 (initial selection):", top_sensors_fd003)


Flat / Constant Sensors to drop: []
Refined Top 5 Variance Sensors in FD003 (initial selection): ['sensor_11', 'sensor_12', 'sensor_7', 'sensor_17', 'sensor_4']


#  Step – Plot Sensor Behavior Across KMeans Cluster Stages (FD003)

In [47]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Load clustered FD003 dataset
df_fd003 = pd.read_csv("../data/clustered_train_FD003.csv")

# Confirm required column exists
if 'kmeans_stage' not in df_fd003.columns:
    raise ValueError("'kmeans_stage' column not found in FD003 dataset.")

# Top 5 sensors from Step 1
top_sensors_fd003 = ['sensor_11', 'sensor_12', 'sensor_7', 'sensor_17', 'sensor_4']

# Create output directory if not exists
output_dir = "../figures/manual_cluster_verification/FD003/"
os.makedirs(output_dir, exist_ok=True)

# Plot each sensor's behavior across KMeans stages
for sensor in top_sensors_fd003:
    plt.figure(figsize=(10, 6))
    sns.lineplot(
        x='time',
        y=sensor,
        hue='kmeans_stage',
        data=df_fd003,
        palette='tab10'
    )
    plt.title(f"{sensor} behavior across KMeans Stages - FD003")
    plt.xlabel("Cycle Number (Time)")
    plt.ylabel(f"{sensor} Reading (Normalized)")
    plt.legend(title='KMeans Stage')
    plt.tight_layout()
    plt.savefig(f"{output_dir}{sensor}_kmeans_stage_FD003.png")
    plt.close()

print(" Sensor behavior plots saved for FD003.")


 Sensor behavior plots saved for FD003.


# Step: FD003 Coinciding Stage Detection Script

In [52]:
import pandas as pd
import itertools

# Load the clustered FD003 dataset
df_fd003 = pd.read_csv("../data/clustered_train_FD003.csv")

# Ensure the clustering column is correct
if 'kmeans_stage' not in df_fd003.columns:
    raise KeyError("Missing 'kmeans_stage' column in FD003 dataset")

# Top 5 sensors selected by variance
top_sensors_fd003 = ['sensor_11', 'sensor_12', 'sensor_7', 'sensor_17', 'sensor_4']

# Calculate and report coinciding stage pairs
coinciding_pairs_fd003 = []

print("===== FD003 Coinciding Sensors and Stage Pairs =====")
for sensor in top_sensors_fd003:
    stage_means = df_fd003.groupby('kmeans_stage')[sensor].mean().sort_index()
    for stage_a, stage_b in itertools.combinations(stage_means.index, 2):
        delta = abs(stage_means[stage_a] - stage_means[stage_b])
        if delta < 0.01:
            coinciding_pairs_fd003.append((sensor, stage_a, stage_b, delta))
            print(f"⚠️ {sensor}: Stage {stage_a} & Stage {stage_b} → Δmean = {delta:.5f}")

if not coinciding_pairs_fd003:
    print(" No coinciding stage pairs detected for FD003. All selected sensors show good separation.")


===== FD003 Coinciding Sensors and Stage Pairs =====
 No coinciding stage pairs detected for FD003. All selected sensors show good separation.


# Step: Add final_stage and Save

In [53]:
# Copy the kmeans_stage column into a new column for final usage
df_fd003['final_stage'] = df_fd003['kmeans_stage']

# Save final top sensors list
final_top_sensors_fd003 = ['sensor_11', 'sensor_12', 'sensor_7', 'sensor_17', 'sensor_4']
print(" Final Top Sensors for FD003 Classification:", final_top_sensors_fd003)

# Save the updated DataFrame
df_fd003.to_csv("../data/corrected_clustered_train_FD003.csv", index=False)
print(" Final version saved as 'corrected_clustered_train_FD003.csv' with 'final_stage' column added.")


 Final Top Sensors for FD003 Classification: ['sensor_11', 'sensor_12', 'sensor_7', 'sensor_17', 'sensor_4']
 Final version saved as 'corrected_clustered_train_FD003.csv' with 'final_stage' column added.


 ---
 # Step : Load FD004 Clustered Data + Get Top 5 Sensors by Variance

In [55]:
import pandas as pd

# Load clustered FD004 dataset
df_fd004 = pd.read_csv("../data/clustered_train_FD004.csv")

# Identify all sensor columns
sensor_columns_fd004 = [col for col in df_fd004.columns if 'sensor_' in col]

# Drop sensors with very low variance
flat_sensors_fd004 = [s for s in sensor_columns_fd004 if df_fd004[s].std() < 0.001]
sensor_columns_fd004 = [s for s in sensor_columns_fd004 if s not in flat_sensors_fd004]

# Compute variance and select top 5
variances_fd004 = df_fd004[sensor_columns_fd004].var().sort_values(ascending=False)
top_sensors_fd004 = variances_fd004.head(5).index.tolist()

print("Flat / Constant Sensors to drop:", flat_sensors_fd004)
print("Refined Top 5 Variance Sensors in FD004 (initial selection):", top_sensors_fd004)


Flat / Constant Sensors to drop: []
Refined Top 5 Variance Sensors in FD004 (initial selection): ['sensor_16', 'sensor_1', 'sensor_19', 'sensor_13', 'sensor_2']


In [56]:
import pandas as pd

# Load clustered FD004 dataset
df_fd004 = pd.read_csv("../data/clustered_train_FD004.csv")

# Identify all sensor columns
sensor_columns_fd004 = [col for col in df_fd004.columns if 'sensor_' in col]

# Drop sensors with very low variance
flat_sensors_fd004 = [s for s in sensor_columns_fd004 if df_fd004[s].std() < 0.001]
sensor_columns_fd004 = [s for s in sensor_columns_fd004 if s not in flat_sensors_fd004]

# Compute variance and select top 5
variances_fd004 = df_fd004[sensor_columns_fd004].var().sort_values(ascending=False)
top_sensors_fd004 = variances_fd004.head(5).index.tolist()

print("Flat / Constant Sensors to drop:", flat_sensors_fd004)
print("Refined Top 5 Variance Sensors in FD004 (initial selection):", top_sensors_fd004)


Flat / Constant Sensors to drop: []
Refined Top 5 Variance Sensors in FD004 (initial selection): ['sensor_16', 'sensor_1', 'sensor_19', 'sensor_13', 'sensor_2']


# Step: Plot Sensor Behavior Across KMeans Stages (FD004)

In [57]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Load clustered FD004 dataset
df_fd004 = pd.read_csv("../data/clustered_train_FD004.csv")

# Safety check
if 'kmeans_stage' not in df_fd004.columns:
    raise ValueError("'kmeans_stage' column not found in FD004 dataset.")

# Top 5 sensors from variance step
top_sensors_fd004 = ['sensor_16', 'sensor_1', 'sensor_19', 'sensor_13', 'sensor_2']

# Output path
output_dir = "../figures/manual_cluster_verification/FD004/"
os.makedirs(output_dir, exist_ok=True)

# Plot sensor behavior
for sensor in top_sensors_fd004:
    plt.figure(figsize=(10, 6))
    sns.lineplot(
        x='time',
        y=sensor,
        hue='kmeans_stage',
        data=df_fd004,
        palette='tab10'
    )
    plt.title(f"{sensor} behavior across KMeans Stages - FD004")
    plt.xlabel("Cycle Number (Time)")
    plt.ylabel(f"{sensor} Reading (Normalized)")
    plt.legend(title='KMeans Stage')
    plt.tight_layout()
    plt.savefig(f"{output_dir}{sensor}_kmeans_stage_FD004.png")
    plt.close()

print(" Sensor behavior plots saved for FD004.")


✅ Sensor behavior plots saved for FD004.


# Step : Coinciding Stage Detection Script for FD004 

In [64]:
import pandas as pd
import itertools

# Load FD004 clustered data
df_fd004 = pd.read_csv("../data/clustered_train_FD004.csv")

# Sensors selected by variance in Step 1
top_sensors_fd004 = ['sensor_16', 'sensor_1', 'sensor_19', 'sensor_13', 'sensor_2']

# Coinciding detection
coinciding_pairs = []

print("===== FD004 Coinciding Sensors and Stage Pairs =====")
for sensor in top_sensors_fd004:
    stage_means = df_fd004.groupby('kmeans_stage')[sensor].mean().sort_index()
    has_overlap = False
    for stage_a, stage_b in itertools.combinations(stage_means.index, 2):
        delta = abs(stage_means[stage_a] - stage_means[stage_b])
        if delta < 0.01:
            has_overlap = True
            coinciding_pairs.append((sensor, stage_a, stage_b, delta))
            print(f"⚠️ {sensor}: Stage {stage_a} & Stage {stage_b} → Δmean = {delta:.5f}")
    if not has_overlap:
        print(f"✅ {sensor}: All stages well-separated")


===== FD004 Coinciding Sensors and Stage Pairs =====
⚠️ sensor_16: Stage Stage 0 & Stage Stage 3 → Δmean = 0.00000
⚠️ sensor_16: Stage Stage 0 & Stage Stage 4 → Δmean = 0.00000
⚠️ sensor_16: Stage Stage 1 & Stage Stage 2 → Δmean = 0.00000
⚠️ sensor_16: Stage Stage 3 & Stage Stage 4 → Δmean = 0.00000
✅ sensor_1: All stages well-separated
⚠️ sensor_19: Stage Stage 0 & Stage Stage 1 → Δmean = 0.00000
⚠️ sensor_19: Stage Stage 0 & Stage Stage 2 → Δmean = 0.00000
⚠️ sensor_19: Stage Stage 0 & Stage Stage 4 → Δmean = 0.00000
⚠️ sensor_19: Stage Stage 1 & Stage Stage 2 → Δmean = 0.00000
⚠️ sensor_19: Stage Stage 1 & Stage Stage 4 → Δmean = 0.00000
⚠️ sensor_19: Stage Stage 2 & Stage Stage 4 → Δmean = 0.00000
⚠️ sensor_13: Stage Stage 0 & Stage Stage 1 → Δmean = 0.00007
⚠️ sensor_13: Stage Stage 0 & Stage Stage 2 → Δmean = 0.00019
⚠️ sensor_13: Stage Stage 0 & Stage Stage 4 → Δmean = 0.00007
⚠️ sensor_13: Stage Stage 1 & Stage Stage 2 → Δmean = 0.00026
⚠️ sensor_13: Stage Stage 1 & Stage Stage

# Step: Finalize Top Sensors for Phase 3 — Drop Misleading Ones

In [65]:
# Initial top sensors from variance ranking
initial_top_sensors_fd004 = ['sensor_16', 'sensor_1', 'sensor_19', 'sensor_13', 'sensor_2']

# Sensors to drop (based on coinciding stage means)
drop_sensors_fd004 = ['sensor_16', 'sensor_19', 'sensor_13']

# Final sensors to retain for Phase 3 Classification
final_top_sensors_fd004 = [s for s in initial_top_sensors_fd004 if s not in drop_sensors_fd004]

print("Initial Top Sensors (based on variance):", initial_top_sensors_fd004)
print("Removed due to poor stage separation:", drop_sensors_fd004)
print("Final Top Sensors for Phase 3 Classification (FD004):", final_top_sensors_fd004)


Initial Top Sensors (based on variance): ['sensor_16', 'sensor_1', 'sensor_19', 'sensor_13', 'sensor_2']
Removed due to poor stage separation: ['sensor_16', 'sensor_19', 'sensor_13']
Final Top Sensors for Phase 3 Classification (FD004): ['sensor_1', 'sensor_2']


# Step: Add final_stage and Save

In [66]:
import pandas as pd

# Load original clustered data
df_fd004 = pd.read_csv("../data/clustered_train_FD004.csv")

# Copy kmeans_stage into final_stage for clean pipeline transition
df_fd004['final_stage'] = df_fd004['kmeans_stage']

# Save the corrected version
df_fd004.to_csv("../data/corrected_clustered_train_FD004.csv", index=False)

print(" Final version saved as corrected_clustered_train_FD004.csv with 'final_stage' column added.")


 Final version saved as corrected_clustered_train_FD004.csv with 'final_stage' column added.
